In [1]:
import pandas as pd
from pathlib import Path

project_folder = Path.cwd()

if project_folder.name == "notebooks":
    project_folder = project_folder.parent

data_folder = project_folder / "data"
feature_file = data_folder / "propertylens_featured.csv"

df = pd.read_csv(feature_file)

# Create locality and property-type benchmarks
benchmark_counts = (
    df.groupby(
        ["locality", "flat_type"],
        observed=True
    )
    .size()
    .reset_index(name="benchmark_listings")
)

benchmarks = (
    df.groupby(
        ["locality", "flat_type"],
        observed=True
    )["rate_per_sqft"]
    .median()
    .reset_index(name="local_median_rate_per_sqft")
)

benchmarks = benchmarks.merge(
    benchmark_counts,
    on=["locality", "flat_type"],
    how="left"
)

# Keep only locality/property-type groups with enough observations
benchmarks = benchmarks[
    benchmarks["benchmark_listings"] >= 20
].copy()

# Match benchmarks to individual listings
df = df.merge(
    benchmarks,
    on=["locality", "flat_type"],
    how="left"
)

# Compare each listing with its local property-type benchmark
df["rate_difference_pct"] = (
    (df["rate_per_sqft"] - df["local_median_rate_per_sqft"])
    / df["local_median_rate_per_sqft"]
    * 100
)

# Classify listings with a valid benchmark
df["value_position"] = pd.NA

valid_benchmark = df["local_median_rate_per_sqft"].notna()

df.loc[
    valid_benchmark & (df["rate_difference_pct"] < -10),
    "value_position"
] = "Lower-Priced"

df.loc[
    valid_benchmark
    & (df["rate_difference_pct"] >= -10)
    & (df["rate_difference_pct"] <= 10),
    "value_position"
] = "Market-Aligned"

df.loc[
    valid_benchmark & (df["rate_difference_pct"] > 10),
    "value_position"
] = "Premium-Priced"


# Benchmark coverage
benchmarked_listings = valid_benchmark.sum()
unbenchmarked_listings = (~valid_benchmark).sum()

print("Benchmark coverage:")
print(f"Listings with a local benchmark: {benchmarked_listings}")
print(f"Listings without a local benchmark: {unbenchmarked_listings}")


# Overall value-position distribution
value_order = [
    "Lower-Priced",
    "Market-Aligned",
    "Premium-Priced"
]

value_position_summary = (
    df["value_position"]
    .value_counts()
    .reindex(value_order)
    .fillna(0)
    .astype(int)
)

value_position_pct = (
    value_position_summary
    .div(benchmarked_listings)
    .mul(100)
    .round(2)
)

value_position_summary = pd.DataFrame({
    "listings": value_position_summary,
    "share_pct": value_position_pct
})

print("\nValue position distribution:")
print(value_position_summary.to_string())


# Locality-level value-position distribution
locality_value_counts = pd.crosstab(
    df["locality"],
    df["value_position"]
).reindex(columns=value_order, fill_value=0)

locality_value_counts["total_benchmarked"] = (
    locality_value_counts.sum(axis=1)
)

# Keep localities with at least 20 benchmarked listings
locality_value_counts = locality_value_counts[
    locality_value_counts["total_benchmarked"] >= 20
].copy()

locality_value_share = (
    locality_value_counts[value_order]
    .div(locality_value_counts["total_benchmarked"], axis=0)
    .mul(100)
    .round(2)
)

locality_value_summary = locality_value_counts[
    ["total_benchmarked"]
].copy()

for column in value_order:
    locality_value_summary[f"{column}_pct"] = locality_value_share[column]

locality_value_summary = locality_value_summary.sort_values(
    "total_benchmarked",
    ascending=False
)

print("\nLocality value-position distribution:")
print(locality_value_summary.head(15).to_string())


# Property type distribution by value position
value_position_property_type = pd.crosstab(
    df["value_position"],
    df["flat_type"]
).reindex(value_order)

print("\nProperty type counts by value position:")
print(
    value_position_property_type
    .fillna(0)
    .astype(int)
    .to_string()
)

Benchmark coverage:
Listings with a local benchmark: 12635
Listings without a local benchmark: 1590

Value position distribution:
                listings  share_pct
value_position                     
Lower-Priced        4182      33.10
Market-Aligned      4729      37.43
Premium-Priced      3724      29.47

Locality value-position distribution:
value_position   total_benchmarked  Lower-Priced_pct  Market-Aligned_pct  Premium-Priced_pct
locality                                                                                    
Sector 65                      854             35.13               20.37               44.50
Sector 79                      568             45.77               32.75               21.48
Sector 63                      475             30.95               52.00               17.05
Sector 67                      463             38.23               58.10                3.67
Sector 113                     447             27.52               42.51               29.98
